# Sensor-space TEPs, one average per block

Python/MNE version of `TEP_channel_TESA.m`. Reads `organize_neurone.m`'s BIDS output and
produces one channel-space TMS-evoked potential per block, for a single subject.

```
organize_neurone.m  ->  THIS NOTEBOOK  ->  per-block TEP curves + figures
 (per-block .set +      (epoch, clean,
  events.tsv)            average)
```

**Why it reads the raw blocks and not `derivatives/preprocessing_prime/`.** That derivative is
prime's own output and covers **1-100 ms only** at 60 channels -- exactly the window the N45
dipole filter needs, and nothing else. A sensor-space TEP needs a pre-stimulus baseline and the
late components (N100, P180), so this notebook epochs the raw recordings itself.

**Blocks.** All left-M1 stimulation (right-hand FDI/APB target). Each is cleaned and averaged completely independently:

| block | trials |
|---|---|
| `baseline` | 100 open-loop single pulses |
| `calibration` | 125 single pulses |
| `intervention_block_1..4`, split by condition | 60 PRIME-triggered + 20 predetermined each |
| `evaluation-t0/t15/t30/t60` | 100 single pulses each |
| `intervention_all` (pooled), split by condition | 240 PRIME + 80 predetermined |

The 100 Hz PRIME triplets are excluded throughout: three pulses 10 ms apart do not give an
interpretable single-pulse TEP.

**The first ~25 ms is not interpretable.** On Pilot002 the deflection between the pulse and
~30 ms is dominated by artifact: it is not an exponential decay (a single exponential fits it at
R2 = 0.07, so `tesa_detrend` does nothing) and it is not muscle either (only 11% of it lies above
100 Hz, so SSP-SIR cannot see it -- estimating its subspace from >100 Hz and projecting it out
made the deflection larger, not smaller). What settles it is that the 35-80 ms values from this
notebook converge on those from an independent SOUND + SSP-SIR pipeline run on the same trials
once `CUT_MS` reaches ~25 ms, and disagree badly below that. So the artifact is simply blanked
out to 25 ms, and the interpolated stretch is shaded grey in every figure. Do not read anything
from it -- the N45 window (38-50 ms) sits safely outside.

**Cleaning** is the standard sensor-space chain, fully automatic and with no interactive step:
blank and interpolate the pulse artifact, downsample, drop bad channels and trials on
data-driven thresholds, band-pass and notch filter, ICA for blinks and TMS-evoked muscle,
average reference, baseline correct. Every threshold is a named constant in Section 1.

Requires only `mne`, `numpy`, `pandas`, `matplotlib`.

## 1. Paths and parameters

The only cell that needs editing. `subject` / `session` move it to another subject; everything
below them is the analysis definition.

In [ ]:
from pathlib import Path

# ---- subject / session ---------------------------------------------------------------
bids_root = Path.home() / "Desktop/Loop/BIDS"       # Windows: Path(r"D:\Linus\Loop\BIDS")     # on the Mac: Path.home() / "Desktop/Loop/BIDS"
subject   = "Pilot001"
session   = "prime"
run       = "01"

out_dir = bids_root / "derivatives" / "TEP_sensorspace" / f"sub-{subject}" / f"ses-{session}"
fig_dir = out_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# ---- what counts as a trial ------------------------------------------------------------
TRIGGER_TYPE  = "A - Stimulation"          # the real TMS trigger in these recordings
EXCLUDE_COND  = ["prime_triplet"]          # 100 Hz triplets: not a single-pulse TEP
EMG_CHANNELS  = ["FDIr", "APBr"]           # dropped -- this is an EEG-only analysis
MONTAGE       = "standard_1005"            # the .set channel positions are not reliable
# The recording is referenced to a physical electrode that is NOT in the file: the montage has
# 60 channels with no FCz and no AFz, the standard actiCAP/EasyCap arrangement of FCz = online
# reference, AFz = ground. Averaging over 60 of the 61 electrodes that actually existed biases
# the average reference and therefore every topography, so the reference is added back as a
# zero-filled channel first -- this is the standard step (EEGLAB pop_reref "refloc", TESA).
# VERIFY against this session's NeurOne Protocol.xml; set to None if it was something else.
REF_CHANNEL   = "FCz"

# ---- epoching and pulse-artifact removal -----------------------------------------------
# The shortest interval between any two pulses in this dataset is 2.510 s, so +/-1.25 s is
# the widest epoch that can never contain a second pulse. Wide is what we want: the 1 Hz
# Butterworth's edge transient then dies out long before the window of interest.
TMIN, TMAX    = -1.25, 1.25                # s
CUT_MS        = (-2.0, 25.0)               # ms, blanked and interpolated across. 25 ms
                                           # is where this recording stops disagreeing
                                           # with a SOUND+SSP-SIR pipeline on the same
                                           # trials -- see the note at the top. NOTHING
                                           # inside this window is real data.
SFREQ         = 1000.0                     # Hz, after downsampling
BASELINE      = (-0.5, -0.01)              # s, applied at the very end

# ---- automatic rejection ---------------------------------------------------------------
# Two bad-channel rules, because they catch opposite failures: a LOUD channel is a variance
# outlier, a DEAD or disconnected one is not (its variance can look ordinary) but it stops
# resembling its neighbours. Pilot002's P4 is the second kind -- its best correlation with any
# other channel is 0.04, and a variance rule alone never sees it. Left in, it contaminates the
# average reference and every topography.
#
# All three rules are measured on FILTERED data -- see clean_block. On unfiltered epochs the
# shared DC offset and slow drift inflate every correlation, which both hides real dropouts
# (P4 reads 0.89 unfiltered against 0.04 filtered) and invents false ones.
BAD_CH_Z      = 4.0                        # robust z of log-variance -> bad channel
BAD_CH_CORR   = 0.50                       # best |corr| with any other channel, over all trials
BAD_CH_FRAC   = 0.30                       # ... or a dropout on this fraction of trials alone,
                                           # which catches a channel that fails only part-way
                                           # through a block instead of throughout
TRIAL_MAD_K   = 5.0                        # peak-to-peak > median + k*MAD -> bad trial
MIN_TRIALS    = 30                         # below this a block average is flagged

# ---- filtering -------------------------------------------------------------------------
L_FREQ, H_FREQ = 1.0, 100.0                # Hz band-pass
NOTCH_BAND     = (48.0, 52.0)              # Hz band-stop for line noise
FILTER_ORDER   = 4                         # 4th-order Butterworth, applied forwards+backwards

# ---- ICA ---------------------------------------------------------------------------------
# Components are classified by ICLabel, a trained classifier, rather than by hand-written
# thresholds. Two reasons: writing the rules out by hand only covers the artifact types you
# thought of (blinks and TMS muscle caught 2 of 58 components here), and TESA's published
# thresholds for the remaining types do not transfer to this montage and pipeline -- applied
# as published they flag 52-62% of components carrying half the variance.
#
# ICLabel requires extended-infomax ICA at full rank, on average-referenced, 1-100 Hz data.
# clean_block below is ordered to satisfy that. Budget ~1 min per block for the fit.
USE_ICA        = True
ICA_SEED       = 42                        # fixed, so a run is reproducible
ICLABEL_PROB   = 0.80                      # drop a non-brain component above this confidence
ICLABEL_KEEP   = ["brain", "other"]        # "other" = unclassifiable, kept on purpose

# ---- PRIME prediction split ------------------------------------------------------------
# prediction_probability comes from trials_intervention.csv, already time-matched onto the real
# NeurOne triggers by organize_neurone.m (verified identical to the CSV). Only prime_single_pulse
# trials carry one -- predetermined and calibration pulses have none by definition.
PRED_COL  = "prediction_probability"       # what PRIME predicted, before the pulse
AMP_COL   = "tep_amplitude"                # what actually happened: PRIME's own online
                                           # percentile rank of the resulting TEP within its
                                           # rolling buffer, bounded [0, 1] (it steps by 1/87).
                                           # 12 of 240 trials have none (postprocessing_failed)
                                           # and are dropped from that split.
PRED_Q    = 0.25                           # quartiles
PRED_WITHIN_BLOCK = True                   # take the quartiles within each intervention block
                                           # before pooling, so "high prediction" cannot just
                                           # mean "whichever block ran higher predictions"

# ---- region of interest and plotting ---------------------------------------------------
ROI       = ["C3", "FC3", "CP3", "C1", "C5"]
ROI_NAME  = "M1l"
PLOT_XLIM = (-100, 200)                        # ms
TOPO_WIN  = (40, 50)                           # ms, the N45 the intervention targets

## 2. Trial table

Reads each block's `events.tsv`, drops the triplets, and builds one `block_label` column with
the same values the rest of this project uses (`baseline`, `calibration`,
`intervention_block_1..4`, `evaluation-t0/t15/t30/t60`) -- the `intervention-all` recording
contains five of them, told apart by its own `stage` column.

The block list at the bottom is what Section 4 loops over. `intervention_all` re-uses exactly
the trials of the four per-block entries, cleaned together in one go: 240 trials instead of 60
makes for a far more stable average, and the 20-trial predetermined blocks in particular are
worth looking at pooled.

Blocks a subject does not have (`evaluation-t60` is absent for Pilot001) are reported and
skipped rather than raising.

In [ ]:
import mne, numpy as np, pandas as pd
mne.set_log_level("ERROR")

try:
    from mne_icalabel import label_components
except ImportError as err:                 # pip install mne-icalabel
    raise ImportError("This notebook classifies ICA components with ICLabel. "
                      "Install it with:  pip install mne-icalabel") from err

eeg_dir = bids_root / f"sub-{subject}" / f"ses-{session}" / "eeg"
TASKS = ["baseline", "intervention-all",
         "evaluation-t0", "evaluation-t15", "evaluation-t30", "evaluation-t60"]

events = {}
for task in TASKS:
    stem = f"sub-{subject}_ses-{session}_task-{task}_run-{run}"
    tsv, st = eeg_dir / f"{stem}_events.tsv", eeg_dir / f"{stem}_eeg.set"
    if not (tsv.exists() and st.exists()):
        print(f"!! not found, skipping: task-{task}")
        continue

    df = pd.read_csv(tsv, sep="\t").fillna("")
    df = df[df["trigger_type_neurone"] == TRIGGER_TYPE].reset_index(drop=True)
    if "condition" not in df:
        df["condition"] = ""
    df["condition"] = df["condition"].astype(str)
    # one file, five blocks: intervention-all carries calibration + the four blocks
    df["block_label"] = df["stage"].astype(str) if task == "intervention-all" else task
    df = df[~df["condition"].isin(EXCLUDE_COND)].reset_index(drop=True)
    events[task] = df
    print(f"task-{task:16s} {len(df):4d} single pulses  "
          f"[{', '.join(f'{k}:{v}' for k, v in df['block_label'].value_counts().items())}]")

# ---- the blocks to analyse: (label, task, block_label(s), condition, prediction split) ----
blocks = [("baseline",    "baseline",         ["baseline"],    "", None),
          ("calibration", "intervention-all", ["calibration"], "", None)]
int_labels = [f"intervention_block_{b}" for b in (1, 2, 3, 4)]
for b, lab in enumerate(int_labels, start=1):
    blocks.append((f"int{b}_prime",  "intervention-all", [lab], "prime_single_pulse", None))
    blocks.append((f"int{b}_predet", "intervention-all", [lab], "predetermined_single", None))
for task in [t for t in events if t.startswith("evaluation-")]:
    blocks.append((task.replace("-", "_"), task, [task], "", None))
blocks.append(("intervention_all_prime",  "intervention-all", int_labels, "prime_single_pulse", None))
blocks.append(("intervention_all_predet", "intervention-all", int_labels, "predetermined_single", None))

# The same PRIME single pulses again, split two ways: by what PRIME predicted beforehand,
# and by the TEP amplitude that actually resulted. The fifth element is
# (column, side, quartiles-within-block).
blocks.append(("prime_pred_high", "intervention-all", int_labels, "prime_single_pulse", (PRED_COL, "high", PRED_WITHIN_BLOCK)))
blocks.append(("prime_pred_low",  "intervention-all", int_labels, "prime_single_pulse", (PRED_COL, "low",  PRED_WITHIN_BLOCK)))
blocks.append(("prime_tep_high",  "intervention-all", int_labels, "prime_single_pulse", (AMP_COL, "high", PRED_WITHIN_BLOCK)))
blocks.append(("prime_tep_low",   "intervention-all", int_labels, "prime_single_pulse", (AMP_COL, "low",  PRED_WITHIN_BLOCK)))

# The top quartile of predictions taken across ALL FOUR intervention blocks at once, rather
# than 15 from each. Because the prediction distribution drifts over the session, this is a
# different set of trials from prime_pred_high above -- it is weighted toward whichever blocks
# ran the higher predictions, so it also carries a block/time contrast. The composition is
# printed below so that contrast is visible rather than hidden.
blocks.append(("prime_pred_high_global", "intervention-all", int_labels, "prime_single_pulse", (PRED_COL, "high", False)))

blocks = [b for b in blocks if b[1] in events]
print(f"\n{len(blocks)} blocks to analyse.")

## 3. Epoching and cleaning

Three functions, and they are the whole pipeline.

`get_raw` opens a recording with `preload=False` and caches the handle, so the 3.4 GB
`intervention-all` file never enters memory whole.

`epoch_block` epochs **one block's trials only**, straight off the on-disk `.fdt`. Doing it per
block rather than per recording is what keeps +/-1.25 s epochs affordable: the largest single
block (240 pooled PRIME pulses) peaks around 1.4 GB, where epoching the whole recording at once
would be 5 GB. The pulse artifact is blanked and interpolated **before** downsampling, so the
anti-aliasing filter never sees it.

`clean_block` runs on those trials alone, so each block gets its own bad channels, its own
rejected trials and its own ICA. Bad channels are caught three ways -- loud, dead, and
intermittent (a dropout on more than `BAD_CH_FRAC` of trials, which is how a channel that fails
part-way through a block presents).

**Every rejection statistic is measured after filtering, and that ordering is load-bearing.** On
unfiltered epochs the shared DC offset and slow drift dominate all channel pairs, so the
correlations sit near 1 and stop discriminating: P4 reads 0.89 unfiltered against 0.04 filtered,
which is why it was missed in `baseline` while being caught elsewhere. The same reorder drops a
false positive (FC4 in `evaluation-t0`) and recovers FT7, which falls to r = 0.28 in intervention
block 4. The weakest three channels are printed per block whether or not they crossed a
threshold, so borderline cases are visible rather than silently kept. Bad channels are caught two ways (variance outlier, and loss of
correlation with every other channel); bad trials by peak-to-peak MAD outlier. Components are
then classified by ICLabel and anything non-brain above `ICLABEL_PROB` is removed.

Before the average reference is applied, the online reference electrode is added back as a
channel of zeros. It was never saved to file, so without this the "average" is taken over 60 of
the 61 electrodes that physically existed, which tilts every topography.

The step order matters and is not the obvious one: filter, then interpolate bad channels, then
average reference, and only then ICA. ICLabel was trained on average-referenced 1-100 Hz data
decomposed by extended infomax at full rank, so it has to see the data in that state. Baseline
correction happens last, after the components are gone.

In [ ]:
_raws = {}

def get_raw(task):
    """One cached, un-preloaded handle per recording."""
    if task not in _raws:
        stem = f"sub-{subject}_ses-{session}_task-{task}_run-{run}"
        r = mne.io.read_raw_eeglab(str(eeg_dir / f"{stem}_eeg.set"), preload=False)
        r.drop_channels([c for c in EMG_CHANNELS if c in r.ch_names])
        r.set_montage(mne.channels.make_standard_montage(MONTAGE), on_missing="warn")
        _raws[task] = r
    return _raws[task]


def epoch_block(task, rows):
    """Epoch just these trials, blank and interpolate the pulse, then downsample."""
    raw = get_raw(task)
    ev = np.column_stack([np.round(rows["onset"].to_numpy(float) * raw.info["sfreq"]).astype(int),
                          np.zeros(len(rows), int), np.ones(len(rows), int)])
    ep = mne.Epochs(raw, ev, event_id={"TMS": 1}, tmin=TMIN, tmax=TMAX, baseline=None,
                    metadata=rows.reset_index(drop=True), preload=True, reject=None)
    ep = mne.preprocessing.fix_stim_artifact(ep, tmin=CUT_MS[0] / 1000, tmax=CUT_MS[1] / 1000,
                                             mode="linear")
    ep.resample(SFREQ)
    return ep


def clean_block(ep):
    """Bad channels, bad trials, filter, ICA, average reference, baseline. Returns Epochs."""
    n_in = len(ep)

    # --- filter FIRST: every rejection statistic below is measured on filtered data.
    #     4th-order Butterworth, zero-phase. MNE's default FIR filter would need a
    #     ~3 s kernel for a 1 Hz high-pass, so IIR it is -- which is also exactly what TESA
    #     does (tesa_filtbutter). l_freq > h_freq = band-stop.
    iir = dict(order=FILTER_ORDER, ftype="butter", output="sos")
    ep.filter(L_FREQ, H_FREQ, method="iir", iir_params=iir, picks="eeg")
    ep.filter(NOTCH_BAND[1], NOTCH_BAND[0], method="iir", iir_params=iir, picks="eeg")

    # --- bad channels: loud (variance outlier), dead (uncorrelated with every other channel
    #     across the whole block), or intermittent (a dropout on BAD_CH_FRAC of trials).
    X = ep.get_data()
    nch = len(ep.ch_names)
    v = np.log(X.var(axis=(0, 2)))
    z = (v - np.median(v)) / (1.4826 * np.median(np.abs(v - np.median(v))) + 1e-30)

    C = np.corrcoef(X.transpose(1, 0, 2).reshape(nch, -1))
    np.fill_diagonal(C, np.nan)
    best = np.nanmax(np.abs(C), axis=1)

    drop = np.zeros(nch)               # fraction of trials on which each channel drops out
    for k in range(X.shape[0]):
        Ck = np.corrcoef(X[k])
        np.fill_diagonal(Ck, np.nan)
        drop += np.nanmax(np.abs(Ck), axis=1) < BAD_CH_CORR
    drop /= X.shape[0]

    bad_mask = (np.abs(z) > BAD_CH_Z) | (best < BAD_CH_CORR) | (drop > BAD_CH_FRAC)
    ep.info["bads"] = sorted(np.array(ep.ch_names)[bad_mask].tolist())

    # Print the three least-connected channels whether or not they crossed a threshold, so a
    # borderline one is visible instead of silently kept.
    print("      weakest channels: " + ", ".join(
        f"{ep.ch_names[i]} (r={best[i]:.2f}, {100*drop[i]:.0f}% of trials)"
        for i in np.argsort(best)[:3]))

    # --- bad trials: peak-to-peak outliers (bad channels excluded, or they would drive it) ---
    good = mne.pick_types(ep.info, eeg=True, exclude="bads")
    worst = np.ptp(X[:, good, :], axis=2).max(axis=1)
    thr = np.median(worst) + TRIAL_MAD_K * 1.4826 * np.median(np.abs(worst - np.median(worst)))
    ep = ep[np.flatnonzero(worst <= thr)]

    # --- interpolate bad channels and average reference (both required by ICLabel) ---
    bads = list(ep.info["bads"])
    ep.interpolate_bads(reset_bads=True)
    if REF_CHANNEL and REF_CHANNEL not in ep.ch_names:
        ep = mne.add_reference_channels(ep, REF_CHANNEL)          # the implicit online reference
        ep.set_montage(mne.channels.make_standard_montage(MONTAGE), on_missing="warn")
    ep.set_eeg_reference("average")

    # --- ICA, classified by ICLabel ---
    n_ic, ic_note, n_comp = 0, "ICA off", 0
    if USE_ICA:
        ica = mne.preprocessing.ICA(n_components=None, method="infomax",
                                    fit_params=dict(extended=True),
                                    random_state=ICA_SEED, max_iter="auto")
        ica.fit(ep, picks="eeg")
        n_comp = ica.n_components_

        lab = label_components(ep, ica, method="iclabel")
        y, prob = np.array(lab["labels"]), np.asarray(lab["y_pred_proba"])
        ica.exclude = np.flatnonzero(~np.isin(y, ICLABEL_KEEP) & (prob > ICLABEL_PROB)).tolist()
        ica.apply(ep)

        n_ic = len(ica.exclude)
        counts = pd.Series(y[ica.exclude]).value_counts()
        ic_note = ", ".join(f"{n} {k}" for k, n in counts.items()) or "none"

    ep.apply_baseline(BASELINE)

    ep.info["temp"] = dict(n_in=n_in, n_kept=len(ep), n_ic=n_ic, n_comp=n_comp,
                           ic_note=ic_note, bads=" ".join(bads))
    return ep

## 4. Run every block

One `epoch_block` + `clean_block` pass per block, each independent of the others. The summary
table is the thing to read afterwards: a block that lost an unusual share of its trials, or
whose ICA removed an unusual number of components, is the one to be suspicious of.

`flag` marks blocks with fewer than `MIN_TRIALS` surviving trials -- with the defaults that is
only the 20-trial predetermined blocks, which is exactly why the pooled `intervention_all_*`
blocks exist.

In [ ]:
results, summary = {}, []
for label, task, blabels, cond, split in blocks:
    df = events[task]
    m = df["block_label"].isin(blabels).to_numpy()
    if cond:
        m = m & (df["condition"] == cond).to_numpy()
    rows = df[m]

    if split:                                  # keep only one quartile of the chosen column
        scol, side, within = split
        rows = rows.copy()
        rows["_split"] = pd.to_numeric(rows[scol], errors="coerce")
        rows = rows[rows["_split"].notna()]
        q = 1 - PRED_Q if side == "high" else PRED_Q
        if within:
            thr = rows.groupby("block_label")["_split"].transform(lambda x: x.quantile(q))
        else:
            thr = rows["_split"].quantile(q)
        rows = rows[rows["_split"] >= thr] if side == "high" else rows[rows["_split"] <= thr]
        comp = rows["block_label"].value_counts().sort_index()
        print(f"  {label:24s} {scol} {side:4s} ({'within' if within else 'across'} blocks): "
              f"{len(rows)} trials, range {rows['_split'].min():.3f}-{rows['_split'].max():.3f}")
        print(f"  {'':24s} from " + ", ".join(f"blk{k[-1]}:{v}" for k, v in comp.items()))

    if len(rows) == 0:
        print(f"  {label:24s} no trials, skipped")
        continue

    ep = clean_block(epoch_block(task, rows))
    info = ep.info["temp"]
    roi_pick = [ep.ch_names.index(c) for c in ROI if c in ep.ch_names]
    roi_trials = ep.get_data()[:, roi_pick, :].mean(axis=1) * 1e6      # (n_trials, n_times)

    results[label] = dict(
        evoked=ep.average(), times=ep.times * 1000,
        split_median=float(rows["_split"].median()) if split else np.nan,
        roi=roi_trials.mean(0), sem=roi_trials.std(0, ddof=1) / np.sqrt(len(ep)),
        n=len(ep))
    summary.append(dict(block=label, n_in=info["n_in"], n_kept=info["n_kept"],
                        trials_dropped=info["n_in"] - info["n_kept"],
                        bad_channels=info["bads"], n_ICs=info["n_ic"],
                        n_components=info["n_comp"], ICs=info["ic_note"],
                        split_on=(split[0] if split else ""),
                        split_median=(round(float(rows["_split"].median()), 3) if split else ""),
                        flag="" if len(ep) >= MIN_TRIALS else "few trials"))
    print(f"  {label:24s} {info['n_kept']:3d}/{info['n_in']:3d} trials, "
          f"bad ch: {info['bads'] or '-'}, "
          f"{info['n_ic']}/{info['n_comp']} ICs removed ({info['ic_note']})")
    del ep

_raws.clear()
summary = pd.DataFrame(summary)
summary.to_csv(out_dir / f"sub-{subject}_ses-{session}_desc-TEPsensor_summary.tsv",
               sep="\t", index=False)
summary

## 5. One figure per block

Four panels each: the butterfly of all channels, the right-M1 ROI with its 95% confidence
interval across trials, the global mean field amplitude, and the scalp topography over the N45
window. Saved to `figures/` and also drawn inline.

In [ ]:
import matplotlib.pyplot as plt

def plot_block(label, r):
    t, s = r["times"], (r["times"] >= PLOT_XLIM[0]) & (r["times"] <= PLOT_XLIM[1])
    data = r["evoked"].data * 1e6
    ci = 1.96 * r["sem"]

    fig, ax = plt.subplots(2, 2, figsize=(12, 7.5))
    ax[0, 0].plot(t[s], data[:, s].T, lw=0.5)
    ax[0, 0].set(title=f"Butterfly, {data.shape[0]} channels", xlabel="Time (ms)",
                 ylabel="Amplitude (uV)")

    ax[0, 1].fill_between(t[s], (r["roi"] - ci)[s], (r["roi"] + ci)[s], alpha=0.25)
    ax[0, 1].plot(t[s], r["roi"][s], lw=2)
    ax[0, 1].set(title=f"ROI {ROI_NAME} ({' '.join(ROI)}), mean +/- 95% CI",
                 xlabel="Time (ms)", ylabel="Amplitude (uV)")

    ax[1, 0].plot(t[s], data[:, s].std(axis=0), "k", lw=1.5)
    ax[1, 0].set(title="Global mean field amplitude", xlabel="Time (ms)", ylabel="GMFA (uV)")

    for a in (ax[0, 0], ax[0, 1], ax[1, 0]):
        a.axvspan(*CUT_MS, color="0.85", zorder=0)      # interpolated: not real data
        a.axvline(0, color="r", ls="--", lw=1)
        a.axhline(0, color="k", ls=":", lw=0.8)

    w = (t >= TOPO_WIN[0]) & (t <= TOPO_WIN[1])
    mne.viz.plot_topomap(data[:, w].mean(axis=1), r["evoked"].info, axes=ax[1, 1], show=False)
    ax[1, 1].set(title=f"Topography {TOPO_WIN[0]}-{TOPO_WIN[1]} ms")

    fig.suptitle(f"sub-{subject} ses-{session}  |  {label}  |  {r['n']} trials")
    fig.tight_layout()
    fig.savefig(fig_dir / f"sub-{subject}_ses-{session}_block-{label}_desc-tep.png", dpi=150)
    plt.show()

for label, r in results.items():
    plot_block(label, r)

## 6. Across blocks

Four views. First the same 240 PRIME-triggered single pulses split two ways: by what PRIME
**predicted** before the pulse (`prediction_probability`), and by the TEP amplitude that
**actually resulted** (`tep_amplitude`, PRIME's own online percentile rank of each TEP). Top
quartile against bottom quartile in each case, with 95% confidence intervals.

The two are worth reading against each other. The achieved-amplitude split is close to a
manipulation check -- it should separate, because it is defined on the outcome -- whereas the
predicted split only separates to the extent that the classifier works. In this session the two
correlate at r = +0.12, so expect the prediction split to be much the weaker of the two. Quartiles are taken **within** each intervention block
before pooling (`PRED_WITHIN_BLOCK`), because the prediction distribution drifts across the
session -- block 3 runs lower than block 2 -- so a single global threshold would partly be
selecting on block rather than on confidence. Predetermined and calibration pulses are excluded
throughout: they carry no prediction by definition.

Then the two original views. First every block's ROI waveform on one axis, which is where a block that came out
unlike the others shows up. Then each intervention condition averaged **across** the four
blocks -- the mean of the four separately-cleaned block averages, with the between-block SEM
shaded and the individual blocks in grey. Compare that against `intervention_all_*` from
Section 4, where the same trials were instead cleaned together in one pass.

In [ ]:
per_block = [b for b in results if not b.startswith("intervention_all")]

fig, ax = plt.subplots(figsize=(11, 5))
for label in per_block:
    r = results[label]
    s = (r["times"] >= PLOT_XLIM[0]) & (r["times"] <= PLOT_XLIM[1])
    ax.plot(r["times"][s], r["roi"][s], lw=1.3, label=f"{label} (n={r['n']})")
ax.axvspan(*CUT_MS, color="0.85", zorder=0)
ax.axvline(0, color="r", ls="--", lw=1); ax.axhline(0, color="k", ls=":", lw=0.8)
ax.set(xlabel="Time (ms)", ylabel="Amplitude (uV)",
       title=f"sub-{subject}: {ROI_NAME} ROI, every block")
ax.legend(fontsize=7, ncol=2, loc="upper right")
fig.tight_layout()
fig.savefig(fig_dir / f"sub-{subject}_ses-{session}_desc-TEPsensor_allblocks.png", dpi=150)
plt.show()

# --- top vs bottom quartile, for both splits ---------------------------------------------
def compare_quartiles(hi_lab, lo_lab, title, fname):
    if not {hi_lab, lo_lab} <= set(results):
        return
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for lab, col in [(lo_lab, "tab:grey"), (hi_lab, "tab:red")]:
        r = results[lab]
        s = (r["times"] >= PLOT_XLIM[0]) & (r["times"] <= PLOT_XLIM[1])
        ci = 1.96 * r["sem"]
        ax.fill_between(r["times"][s], (r["roi"] - ci)[s], (r["roi"] + ci)[s],
                        color=col, alpha=0.20, lw=0)
        ax.plot(r["times"][s], r["roi"][s], color=col, lw=2,
                label=f"{lab} (n={r['n']}, median {r['split_median']:.3f})")
    ax.axvspan(*CUT_MS, color="0.85", zorder=0)
    ax.axvline(0, color="r", ls="--", lw=1); ax.axhline(0, color="k", ls=":", lw=0.8)
    ax.set(xlabel="Time (ms)", ylabel="Amplitude (uV)", xlim=PLOT_XLIM,
           title=f"sub-{subject}: {ROI_NAME} ROI, {title}")
    ax.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(fig_dir / f"sub-{subject}_ses-{session}_desc-TEPsensor_{fname}.png", dpi=150)
    plt.show()

    t = results[hi_lab]["times"]
    hi, lo = results[hi_lab]["roi"], results[lo_lab]["roi"]
    sem = np.sqrt(results[hi_lab]["sem"]**2 + results[lo_lab]["sem"]**2)
    print(f"{title}\n{'window':>14}{'top':>12}{'bottom':>12}{'difference':>16}")
    for a_, b_, nm in [(35, 55, "N45 35-55"), (55, 80, "55-80"), (85, 140, "85-140")]:
        w = (t >= a_) & (t <= b_)
        d, e = hi[w].mean() - lo[w].mean(), sem[w].mean()
        print(f"{nm:>14}{hi[w].mean():10.2f} uV{lo[w].mean():10.2f} uV"
              f"{d:9.2f} +/- {e:.2f} uV")
    print()

compare_quartiles("prime_pred_high", "prime_pred_low",
                  "PRIME single pulses by PREDICTED probability", "predictionQuartiles")
compare_quartiles("prime_tep_high", "prime_tep_low",
                  "PRIME single pulses by ACHIEVED TEP amplitude", "tepAmplitudeQuartiles")

groups = {"PRIME single pulses":        [f"int{b}_prime"  for b in (1, 2, 3, 4)],
          "Predetermined single pulses": [f"int{b}_predet" for b in (1, 2, 3, 4)]}

grand = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for a, (name, labels) in zip(axes, groups.items()):
    labels = [l for l in labels if l in results]
    stack = np.vstack([results[l]["roi"] for l in labels])
    t = results[labels[0]]["times"]
    s = (t >= PLOT_XLIM[0]) & (t <= PLOT_XLIM[1])
    mean, sem = stack.mean(0), stack.std(0, ddof=1) / np.sqrt(len(labels))
    grand[name] = dict(times=t, mean=mean, sem=sem,
                       n_trials=sum(results[l]["n"] for l in labels))

    a.plot(t[s], stack[:, s].T, color="0.75", lw=0.7)
    a.fill_between(t[s], (mean - sem)[s], (mean + sem)[s], alpha=0.25)
    a.plot(t[s], mean[s], lw=2)
    a.axvspan(*CUT_MS, color="0.85", zorder=0)
    a.axvline(0, color="r", ls="--", lw=1); a.axhline(0, color="k", ls=":", lw=0.8)
    a.set(title=f"{name}\n{len(labels)} blocks, {grand[name]['n_trials']} trials",
          xlabel="Time (ms)", ylabel="Amplitude (uV)")
fig.suptitle(f"sub-{subject}: {ROI_NAME} ROI averaged across intervention blocks")
fig.tight_layout()
fig.savefig(fig_dir / f"sub-{subject}_ses-{session}_desc-TEPsensor_interventionGrandAverage.png",
            dpi=150)
plt.show()

## 7. Save the waveforms

One tidy CSV of every block's ROI waveform (plus its CI and GMFA), one for the across-block
grand averages, and the evoked responses themselves as a standard `-ave.fif` that
`mne.read_evokeds` reads back with all 60 channels intact -- so any later analysis, topography
or peak measurement can start from this rather than re-running the notebook.

In [ ]:
rows = []
for label, r in results.items():
    rows.append(pd.DataFrame({"block": label, "time_ms": r["times"], "roi_uV": r["roi"],
                              "ci95_uV": 1.96 * r["sem"],
                              "gmfa_uV": (r["evoked"].data * 1e6).std(axis=0),
                              "n_trials": r["n"]}))
waveforms = pd.concat(rows, ignore_index=True)
waveforms.to_csv(out_dir / f"sub-{subject}_ses-{session}_desc-TEPsensor_waveforms.csv",
                 index=False)

pd.concat([pd.DataFrame({"condition": k, "time_ms": g["times"],
                         "roi_uV": g["mean"], "sem_uV": g["sem"]})
           for k, g in grand.items()], ignore_index=True).to_csv(
    out_dir / f"sub-{subject}_ses-{session}_desc-TEPsensor_interventionGrandAverage.csv",
    index=False)

evokeds = []
for label, r in results.items():
    e = r["evoked"].copy()
    e.comment = label
    evokeds.append(e)
mne.write_evokeds(out_dir / f"sub-{subject}_ses-{session}_desc-TEPsensor_ave.fif",
                  evokeds, overwrite=True)

print(f"Saved {len(results)} block averages to:\n  {out_dir}")
waveforms.head()